In [1]:
import pandas as pd
df = pd.read_parquet('data/retail_preprocessed.parquet')

### 고객 구매 빈도와 재구매 간격 확인

고객별 주문 횟수, 재구매 고객 비율, 주문 간격을 확인하여 주별 예측의 적절성을 판단한다.

In [6]:
# 정상 구매 거래
purchase_df = df[df['is_purchase']].copy()

# 1. 고객별 주문 횟수
orders = (
    purchase_df
    .groupby('Customer ID')['Invoice']
    .nunique()
)

print('=== 고객별 주문 횟수 ===')
print(orders.describe())


# 2. 고객별 구매 이력
customer_summary = (
    purchase_df
    .groupby('Customer ID')
    .agg(
        orders=('Invoice', 'nunique'),
        first_date=('InvoiceDate', 'min'),
        last_date=('InvoiceDate', 'max')
    )
)

one_time_rate = (customer_summary['orders'] == 1).mean()
repeat_rate = (customer_summary['orders'] >= 2).mean()

print('\n=== 재구매 고객 비율 ===')
print(f'1회 구매 고객 비율: {one_time_rate:.2%}')
print(f'2회 이상 구매 고객 비율: {repeat_rate:.2%}')


# 3. 고객별 주문 간격
invoice_dates = (
    purchase_df
    .drop_duplicates(['Customer ID', 'Invoice'])
    .sort_values(['Customer ID', 'InvoiceDate'])
)

invoice_dates['gap_days'] = (
    invoice_dates
    .groupby('Customer ID')['InvoiceDate']
    .diff()
    .dt.days
)

print('\n=== 주문 간격(일) ===')
print(invoice_dates['gap_days'].describe())


=== 고객별 주문 횟수 ===
count    5852.000000
mean        6.253247
std        12.749286
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       373.000000
Name: Invoice, dtype: float64

=== 재구매 고객 비율 ===
1회 구매 고객 비율: 27.65%
2회 이상 구매 고객 비율: 72.35%

=== 주문 간격(일) ===
count    30742.000000
mean        51.672533
std         75.870189
min          0.000000
25%          7.000000
50%         25.000000
75%         62.000000
max        714.000000
Name: gap_days, dtype: float64


주문 간격(일)의 중앙값이 25 / 평균 51로 4주 기간의 주별 누적 재구매율 예측을 시도할 가치가 있다.